<a href="https://colab.research.google.com/github/Abraham353/Laboratorio_07-MD/blob/develop/Laboratorio_07_MD.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LABORATORIO 7:REGRESIÓN LOGÍSTICA. MÁQUINAS DE VECTOR DE SOPORTE

## INTEGRANTES:
- HUAMAN QUISPE ABRAHAM SEBASTIAN JONATHAN

### A.-Calcule el information value (IV), tanto para el grupo de variable numéricas como categóricas y excluya las que tenga un poder predictivo débil o menor. Además, separe la variable de clasificación del resto de variables para luego obtener los datos de entrenamiento y prueba, tomando de este último el 25% de datos.


In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from scipy.stats import chi2_contingency

# Cargar datos
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/breast-cancer-wisconsin/wdbc.data"
data = pd.read_csv(url, header=None)
data.columns = ['id', 'diagnosis'] + [f'feature_{i}' for i in range(1, 31)]
data['diagnosis'] = data['diagnosis'].map({'M':1, 'B':0})

# Función IV simplificada
def calc_iv(feature, target):
    bins = pd.qcut(feature, q=5, duplicates='drop')
    table = pd.crosstab(bins, target)
    chi2, _, _, _ = chi2_contingency(table)
    return chi2

# Calcular IV para todas las features
iv = {col:calc_iv(data[col], data['diagnosis']) for col in data.columns[2:]}
selected = [k for k,v in iv.items() if v > 100]  # Umbral empírico

# Dividir datos
X = data[selected]
y = data['diagnosis']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

### B.- Genere el modelo de regresión logística y evalúe la exclusión de variables mediante la significancia de los coeficientes. Además, calcule las métricas de clasificación que se implementaron en la parte práctica e interprete sus resultados más importantes.

In [2]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

# Modelo y predicción
logreg = LogisticRegression(max_iter=1000)
logreg.fit(X_train, y_train)
y_pred = logreg.predict(X_test)

# Métricas
print("Regresión Logística:")
print(classification_report(y_test, y_pred))
print("Coeficientes:", dict(zip(selected, logreg.coef_[0])))

Regresión Logística:
              precision    recall  f1-score   support

           0       0.98      0.98      0.98        89
           1       0.96      0.96      0.96        54

    accuracy                           0.97       143
   macro avg       0.97      0.97      0.97       143
weighted avg       0.97      0.97      0.97       143

Coeficientes: {'feature_1': np.float64(-2.2287089744001043), 'feature_2': np.float64(-0.19147727816518034), 'feature_3': np.float64(0.31762660975882495), 'feature_4': np.float64(-0.012955453044874608), 'feature_6': np.float64(0.3155341480931909), 'feature_7': np.float64(0.6461972634438157), 'feature_8': np.float64(0.5887013188731645), 'feature_11': np.float64(0.09086318319037666), 'feature_13': np.float64(-0.6080882605992485), 'feature_14': np.float64(0.1413285515865713), 'feature_17': np.float64(-0.2775746698203928), 'feature_18': np.float64(0.042398695190373516), 'feature_21': np.float64(-0.8069826066331424), 'feature_22': np.float64(0.380708

/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


### C.- Genere el modelo SVM, calcule sus métricas de clasificación y compárelas con las del modelo de regresión logística para ver si hubo o no mejoras.

In [4]:
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score

# Modelo SVM con escalado
svm = Pipeline([('scaler', StandardScaler()), ('svm', SVC(probability=True))])
svm.fit(X_train, y_train)
y_pred_svm = svm.predict(X_test)

# Métricas y comparación
print("\nSVM:")
print(classification_report(y_test, y_pred_svm))
print("\nComparación Accuracy:")
print(f"LogReg: {accuracy_score(y_test, y_pred):.4f}")
print(f"SVM: {accuracy_score(y_test, y_pred_svm):.4f}")


SVM:
              precision    recall  f1-score   support

           0       0.98      0.98      0.98        89
           1       0.96      0.96      0.96        54

    accuracy                           0.97       143
   macro avg       0.97      0.97      0.97       143
weighted avg       0.97      0.97      0.97       143


Comparación Accuracy:
LogReg: 0.9720
SVM: 0.9720
